# 02. v1 헤비 모델 학습 (T4 가능)

비디오 **R(2+1)D-18**(Kinetics 사전학습, Tran et al. CVPR 2018) + 오디오 **log-mel CNN**
(PANNs 스타일, Kong et al. 2020) late fusion → 11개 sigmoid 멀티라벨.
부분 라벨(빈칸)은 손실에서 마스킹. 약라벨(D2/D8)은 수동 라벨이 없는 칸만 채웁니다.

인용: 상세는 `detection/README.md` §4.

In [ ]:
!pip -q install decord av scikit-learn

In [ ]:
# 경로 설정 + Drive 마운트 (Colab)
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE = Path('/content/drive/MyDrive/BabyMon/dataset')   # ← 업로드 위치에 맞게 수정
assert BASE.exists(), f'{BASE} 없음 — Drive 업로드 위치를 확인하세요'

# 클립 폴더 자동 탐색 (clips/ 또는 resized_2/ 또는 BASE 바로 아래)
CLIPS = None
for cand in (BASE/'clips', BASE/'resized_2', BASE):
    if cand.is_dir() and next(cand.glob('*.mp4'), None):
        CLIPS = cand; break
assert CLIPS, f'{BASE} 아래에서 mp4 폴더를 못 찾음 (clips/ 또는 resized_2/)'
assert (BASE/'manifest.csv').exists(), \
    f'{BASE}/manifest.csv 없음 — 로컬 D:\\carved\\dataset\\manifest.csv 와 labels.csv 를 이 폴더로 업로드하세요'
WORK = Path('/content/work'); WORK.mkdir(exist_ok=True)
print('clips:', CLIPS, '/', len(list(CLIPS.glob("*.mp4"))), '개')

In [ ]:
import torch, numpy as np, pandas as pd
LABEL_COLS = ["D1_nose_covered","D2_moving_freq","D3_eyes_open","D4_hands_out",
              "D5_pre_cry","D6_spit_up","D7_crying","D8_baby_sound","D9_mouthing",
              "C1_adult_hand","C2_baby_absent","C3_other_sound"]
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
T_FRAMES, SIZE = 16, 112          # R(2+1)D-18 기본 해상도
BATCH, EPOCHS, LR = 8, 15, 3e-4
SR, N_MELS = 8000, 64

man = pd.read_csv(BASE/'manifest.csv')
lab = pd.read_csv(BASE/'labels.csv', dtype=str).drop_duplicates('file', keep='last')

# 라벨 행렬: 수동 라벨 우선, 빈칸은 약라벨로 보충, 그래도 없으면 NaN(마스킹)
df = man.merge(lab, on='file', how='left')
Y = pd.DataFrame(index=df.index, columns=LABEL_COLS, dtype=float)
for c in LABEL_COLS:
    Y[c] = pd.to_numeric(df.get(c), errors='coerce')
Y.loc[Y['D2_moving_freq'].isna(), 'D2_moving_freq'] = pd.to_numeric(df['weak_D2_moving'], errors='coerce')
Y.loc[Y['D8_baby_sound'].isna(), 'D8_baby_sound'] = pd.to_numeric(df['weak_D8_sound'], errors='coerce')
keep = Y.notna().any(axis=1)      # 라벨이 하나라도 있는 클립만 학습
df, Y = df[keep].reset_index(drop=True), Y[keep].reset_index(drop=True)
print('학습 표본:', len(df), ' (train', (df.split=='train').sum(), '/ val', (df.split=='val').sum(), ')')
print('라벨별 확정 수:\n', Y.notna().sum())

In [ ]:
# 클립 로더: decord 로 균일 T프레임 + av 로 오디오 → log-mel
import decord, av, torchaudio

mel = torchaudio.transforms.MelSpectrogram(SR, n_fft=512, hop_length=256, n_mels=N_MELS)

def load_video(path, t=T_FRAMES, size=SIZE):
    vr = decord.VideoReader(str(path))
    ix = np.linspace(0, len(vr)-1, t).astype(int)
    fr = torch.from_numpy(vr.get_batch(ix).asnumpy()).float()/255.   # T,H,W,3
    fr = fr.permute(3,0,1,2)                                          # 3,T,H,W
    # letterbox → size x size
    _,_,h,w = fr.shape
    s = size/max(h,w)
    fr = torch.nn.functional.interpolate(fr, scale_factor=s, mode='bilinear', align_corners=False)
    pad_h, pad_w = size-fr.shape[2], size-fr.shape[3]
    fr = torch.nn.functional.pad(fr, (pad_w//2, pad_w-pad_w//2, pad_h//2, pad_h-pad_h//2))
    return fr

def load_audio(path, sec=8.0):
    try:
        c = av.open(str(path)); s = c.streams.audio[0]
        r = av.audio.resampler.AudioResampler('s16', 'mono', SR)
        buf = []
        for fr in c.decode(s):
            for f2 in r.resample(fr):
                buf.append(f2.to_ndarray().flatten())
        x = torch.from_numpy(np.concatenate(buf)).float()/32768.
    except Exception:
        x = torch.zeros(int(SR*sec))
    x = torch.nn.functional.pad(x, (0, max(0, int(SR*sec)-len(x))))[:int(SR*sec)]
    return torch.log(mel(x)+1e-6).unsqueeze(0)                        # 1,64,T'

class ClipDS(torch.utils.data.Dataset):
    def __init__(self, dfr, yfr):
        self.f = dfr['file'].tolist()
        self.y = torch.tensor(yfr.values, dtype=torch.float)          # NaN 포함
    def __len__(self): return len(self.f)
    def __getitem__(self, i):
        p = CLIPS/self.f[i]
        return load_video(p), load_audio(p), self.y[i]

tr = ClipDS(df[df.split=='train'], Y[df.split=='train'])
va = ClipDS(df[df.split=='val'],   Y[df.split=='val'])
dl_tr = torch.utils.data.DataLoader(tr, BATCH, shuffle=True,  num_workers=2)
dl_va = torch.utils.data.DataLoader(va, BATCH, shuffle=False, num_workers=2)

In [ ]:
# 모델: R(2+1)D-18 + 오디오 CNN → late fusion
import torchvision

class AudioCNN(torch.nn.Module):
    def __init__(self, dim=128):
        super().__init__()
        ch = [1,32,64,128,128]
        self.blocks = torch.nn.Sequential(*[torch.nn.Sequential(
            torch.nn.Conv2d(ch[i], ch[i+1], 3, padding=1), torch.nn.BatchNorm2d(ch[i+1]),
            torch.nn.ReLU(), torch.nn.MaxPool2d(2)) for i in range(4)])
        self.fc = torch.nn.Linear(128, dim)
    def forward(self, x):
        h = self.blocks(x).mean(dim=(2,3))
        return self.fc(h)

class HeavyAV(torch.nn.Module):
    def __init__(self, n_out=len(LABEL_COLS)):
        super().__init__()
        v = torchvision.models.video.r2plus1d_18(weights='KINETICS400_V1')
        v.fc = torch.nn.Linear(v.fc.in_features, 256)
        self.video, self.audio = v, AudioCNN(128)
        self.head = torch.nn.Sequential(torch.nn.ReLU(), torch.nn.Dropout(0.3),
                                        torch.nn.Linear(256+128, n_out))
    def forward(self, v, a):
        return self.head(torch.cat([self.video(v), self.audio(a)], dim=1))

model = HeavyAV().to(DEV)

def masked_bce(logit, y):
    m = ~torch.isnan(y)
    return torch.nn.functional.binary_cross_entropy_with_logits(
        logit[m], y[m]) if m.any() else logit.sum()*0

In [ ]:
# 학습 (AMP) + 라벨별 AP 평가
from sklearn.metrics import average_precision_score
opt = torch.optim.AdamW(model.parameters(), lr=LR)
scaler = torch.cuda.amp.GradScaler()
best = 0
for ep in range(EPOCHS):
    model.train()
    for v, a, y in dl_tr:
        v, a, y = v.to(DEV), a.to(DEV), y.to(DEV)
        with torch.cuda.amp.autocast():
            loss = masked_bce(model(v, a), y)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    model.eval(); P, T = [], []
    with torch.no_grad():
        for v, a, y in dl_va:
            P.append(torch.sigmoid(model(v.to(DEV), a.to(DEV))).cpu()); T.append(y)
    P, T = torch.cat(P).numpy(), torch.cat(T).numpy()
    aps = {}
    for i, c in enumerate(LABEL_COLS):
        m = ~np.isnan(T[:, i])
        if m.sum() and len(set(T[m, i])) > 1:
            aps[c] = average_precision_score(T[m, i], P[m, i])
    mAP = np.mean(list(aps.values())) if aps else 0
    print(f'ep{ep+1}: loss={loss.item():.4f} mAP={mAP:.3f}', {k: round(v,3) for k,v in aps.items()})
    if mAP > best:
        best = mAP
        torch.save(model.state_dict(), BASE/'heavy_best.pt')
print('best mAP', best, '→', BASE/'heavy_best.pt')

In [ ]:
# (증류 준비) 전체 클립에 대한 교사 로짓 저장 → 03 노트북이 사용
model.load_state_dict(torch.load(BASE/'heavy_best.pt')); model.eval()
allds = ClipDS(df, Y)
dl = torch.utils.data.DataLoader(allds, BATCH, shuffle=False, num_workers=2)
logits = []
with torch.no_grad():
    for v, a, _ in dl:
        logits.append(model(v.to(DEV), a.to(DEV)).cpu())
np.save(BASE/'teacher_logits.npy', torch.cat(logits).numpy())
df['file'].to_csv(BASE/'teacher_files.csv', index=False)
print('saved teacher logits', len(df))